# Binary Classification: AD vs MCI
## 3D CNN for AD vs MCI Classification

이 노트북은 3D CNN을 사용하여 뇌 MRI에서 알츠하이머병(AD)과 경도인지장애(MCI)를 분류하는 이진 분류 모델을 학습합니다.

**데이터**: 
- MRI 이미지: `/Users/othree/Cognitive Reserve Modeling/Data/ADNI_Download`
- CSV 메타데이터: `/Users/othree/Cognitive Reserve Modeling/Data/ADNI_master_merged_12-17-2025.csv`

**분류 클래스**: AD vs MCI (2-way classification)

---

## 1. 라이브러리 설치 및 Import

In [78]:
# # 필요한 라이브러리 설치
# !pip install nibabel scipy pandas pyyaml scikit-learn scikit-image matplotlib progress

In [79]:
# Import statements
import os
import sys
import shutil
import time
import math
import random
import warnings
import traceback
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
import scipy
import scipy.ndimage
from scipy.ndimage.interpolation import shift, rotate, zoom
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import cat
import torch.nn.init as init
import torch.utils.data as data
from torch.optim.lr_scheduler import MultiStepLR
from torch.autograd import Variable, Function

import nibabel as nib
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, roc_curve, auc, confusion_matrix
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.9.1
CUDA available: False
Using device: cpu


/var/folders/zb/yqyg0k1554l0vj7zf6mzbm240000gn/T/ipykernel_83857/1097701886.py:17: DeprecationWarning: Please import `shift` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.interpolation import shift, rotate, zoom
/var/folders/zb/yqyg0k1554l0vj7zf6mzbm240000gn/T/ipykernel_83857/1097701886.py:17: DeprecationWarning: Please import `rotate` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.interpolation import shift, rotate, zoom
/var/folders/zb/yqyg0k1554l0vj7zf6mzbm240000gn/T/ipykernel_83857/1097701886.py:17: DeprecationWarning: Please import `zoom` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.interpolation import shift, rotate, zoom


## 2. Configuration 설정

**✅ 경로가 이미 실제 데이터로 설정되어 있습니다!**

In [ ]:
cfg = {
    'file_name': './saved_model/ad_vs_mci_training',
    'exp_name': 'ad_vs_mci_experiment',
    
    'data': {
        'base_path': '/Users/othree/Cognitive Reserve Modeling/Data',
        'csv_file': '/Users/othree/Cognitive Reserve Modeling/Data/ADNI_master_merged_12-17-2025.csv',
        'batch_size': 4,
        'val_batch_size': 2,
        'workers': 0,
        'train_split': 0.7,
        'val_split': 0.15,
        'test_split': 0.15,
        'percentage_usage': 1.0,
        'classes': ['MCI', 'AD', 'Dementia']  # Binary classification: AD vs MCI
    },
    
    'model': {
        'arch': 'ours',
        'input_channel': 1,
        'nhid': 512,
        'feature_dim': 1024,
        'n_label': 2,  # Binary classification
        'expansion': 8,
        'num_blocks': 0,
        'type_name': 'conv3x3x3',
        'norm_type': 'Instance'
    },
    
    'adv_model': {
        'nhid': 36,
        'out_dim': 12
    },
    
    'training_parameters': {
        'use_age': True,
        'pretrain': None,
        'start_epoch': 0,
        'epochs': 100,
        'print_freq': 10,
        'max_grad_l2_norm': 1.0
    },
    
    'optimizer': {
        'method': 'SGD',
        'par': {
            'lr': 0.001,
            'weight_decay': 0.0
        }
    },
    
    'visdom': {
        'port': None,
        'server': None
    }
}

os.makedirs('./saved_model', exist_ok=True)

print("Configuration loaded:")
print(f"  - Classification: AD vs MCI (Binary)")
print(f"  - Base Path: {cfg['data']['base_path']}")
print(f"  - CSV File: {cfg['data']['csv_file']}")
print(f"  - Classes: {cfg['data']['classes']}")
print(f"  - n_label: {cfg['model']['n_label']}")
print(f"  - Epochs: {cfg['training_parameters']['epochs']}")
print(f"  - Batch size: {cfg['data']['batch_size']}")
print(f"  - Expansion: {cfg['model']['expansion']}")
print(f"  - Learning rate: {cfg['optimizer']['par']['lr']}")
print(f"  - Use age encoding: {cfg['training_parameters']['use_age']}")

## 3. 유틸리티 함수들

In [81]:
# ===== AverageMeter =====
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


# ===== FullModel for DataParallel =====
class FullModel(nn.Module):
    def __init__(self, model, loss):
        super(FullModel, self).__init__()
        self.model = model
        self.loss = loss
    
    def forward(self, inputs, targets):
        outputs = self.model(inputs[0], inputs[1])
        loss = self.loss(outputs, targets)
        return torch.unsqueeze(loss, 0), outputs


def DataParallel_withLoss(model, loss, **kwargs):
    """Wrap model with loss for multi-GPU training"""
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
    model = FullModel(model, loss)
    device_ids = kwargs.get('device_ids', None)
    output_device = kwargs.get('output_device', None)
    model = torch.nn.DataParallel(model, device_ids=device_ids, output_device=output_device).to(device)
    return model


def clip_gradients(myModel, i_iter, max_grad_l2_norm):
    """Clip gradients to prevent explosion"""
    if max_grad_l2_norm is not None:
        nn.utils.clip_grad_norm_(myModel.parameters(), max_grad_l2_norm)


def balanced_accuracy_score(y_true, y_pred, sample_weight=None, adjusted=False):
    """Calculate balanced accuracy (평균 per-class accuracy)"""
    C = confusion_matrix(y_true, y_pred, sample_weight=sample_weight)
    with np.errstate(divide='ignore', invalid='ignore'):
        per_class = np.diag(C) / C.sum(axis=1)
    if np.any(np.isnan(per_class)):
        warnings.warn('y_pred contains classes not in y_true')
        per_class = per_class[~np.isnan(per_class)]
    score = np.mean(per_class)
    if adjusted:
        n_classes = len(per_class)
        chance = 1 / n_classes
        score -= chance
        score /= 1 - chance
    return score


def get_auc_data(logit_all, target_all, n_label):
    """Calculate ROC-AUC for multi-class classification"""
    y = label_binarize(target_all, classes=list(range(n_label)))
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    # Per-class AUC
    for k in range(n_label):
        fpr[k], tpr[k], _ = roc_curve(y[:, k], logit_all[:, k])
        roc_auc[k] = auc(fpr[k], tpr[k])
    
    # Micro-average AUC
    fpr["micro"], tpr["micro"], _ = roc_curve(y.ravel(), logit_all.ravel())
    roc_auc[k+1] = auc(fpr["micro"], tpr["micro"])
    
    # Macro-average AUC
    all_fpr = np.unique(np.concatenate([fpr[k] for k in range(n_label)]))
    mean_tpr = np.zeros_like(all_fpr)
    for k in range(n_label):
        mean_tpr += np.interp(all_fpr, fpr[k], tpr[k])
    mean_tpr /= n_label
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc[k+2] = auc(fpr["macro"], tpr["macro"])
    
    # Prepare plotting data
    plotting_fpr = [fpr[k] for k in range(n_label)] + [fpr["micro"], fpr["macro"]]
    plotting_tpr = [tpr[k] for k in range(n_label)] + [tpr["micro"], tpr["macro"]]
    
    return plotting_fpr, plotting_tpr, roc_auc


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)
    
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    
    res = []
    correct_all = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0, keepdim=True)
        correct_all.append(correct_k.clone())
        res.append(correct_k.mul_(100.0 / batch_size))
    return res, correct_all

print("✓ Utility functions loaded")

✓ Utility functions loaded


## 4. Loss Functions

In [82]:
def get_loss_criterion(loss_config, type):
    """Get loss function based on type"""
    if type == 'CrossEntropyLoss':
        loss_criterion = nn.CrossEntropyLoss(ignore_index=-100)
    elif type == 'mse':
        loss_criterion = nn.MSELoss()
    else:
        raise NotImplementedError(f"Loss type '{type}' not implemented")
    return loss_criterion

print("✓ Loss functions loaded")

✓ Loss functions loaded


## 5. Data Augmentation Functions

In [83]:
# 3D Image augmentation functions

def translateit(image, offset, isseg=False):
    """3D translation"""
    order = 0 if isseg else 5
    return shift(image, (int(offset[0]), int(offset[1]), int(offset[2])), order=order, mode='constant')


def scaleit(image, factor, isseg=False):
    """3D scaling"""
    order = 0 if isseg else 3
    height, width, depth = image.shape
    zheight = int(np.round(factor * height))
    zwidth = int(np.round(factor * width))
    zdepth = depth
    
    if factor < 1.0:
        newimg = np.zeros_like(image)
        row = (height - zheight) // 2
        col = (width - zwidth) // 2
        layer = (depth - zdepth) // 2
        newimg[row:row+zheight, col:col+zwidth, layer:layer+zdepth] = zoom(
            image, (float(factor), float(factor), 1.0), order=order, mode='nearest'
        )[0:zheight, 0:zwidth, 0:zdepth]
        return newimg
    elif factor > 1.0:
        row = (zheight - height) // 2
        col = (zwidth - width) // 2
        layer = (zdepth - depth) // 2
        newimg = zoom(
            image[row:row+zheight, col:col+zwidth, layer:layer+zdepth],
            (float(factor), float(factor), 1.0), order=order, mode='nearest'
        )
        extrah = (newimg.shape[0] - height) // 2
        extraw = (newimg.shape[1] - width) // 2
        extrad = (newimg.shape[2] - depth) // 2
        newimg = newimg[extrah:extrah+height, extraw:extraw+width, extrad:extrad+depth]
        return newimg
    else:
        return image


def rotateit(image, axes, theta, isseg=False):
    """3D rotation"""
    order = 0 if isseg else 5
    return rotate(image, float(theta), axes=axes, reshape=False, order=order, mode='constant')


def flipit(image):
    """Random flipping"""
    lr_thred = np.random.uniform(0, 1, 1)[0]
    ud_thred = np.random.uniform(0, 1, 1)[0]
    if lr_thred <= 0.5:
        image = np.fliplr(image)
    if ud_thred >= 0.5:
        image = np.flipud(image)
    return image


def intensifyit(image, factor):
    """Intensity adjustment"""
    return image * float(factor)


def resampleit(image, dims, isseg=False):
    """Resampling"""
    order = 0 if isseg else 5
    image = zoom(image, np.array(dims)/np.array(image.shape, dtype=np.float32), 
                 order=order, mode='nearest')
    if image.shape[-1] == 3:  # RGB image
        return image
    else:
        return image if isseg else (image - image.min()) / (image.max() - image.min())

print("✓ Augmentation functions loaded")

✓ Augmentation functions loaded


## 6. Model Architecture

In [84]:
class AgeEncoding(nn.Module):
    """Age encoding using positional encoding"""
    def __init__(self, d_model, dropout, out_dim, max_len=240):
        super(AgeEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp((torch.arange(0, d_model, 2).float() *
                             -(math.log(10000.0) / d_model)).float())
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        
        self.fc6 = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.LayerNorm(512),
            nn.Linear(512, out_dim)
        )
    
    def forward(self, x, age_id):
        y = torch.autograd.Variable(self.pe[age_id, :], requires_grad=False)
        y = self.fc6(y)
        x += y
        return self.dropout(x)


class NetWork(nn.Module):
    """3D CNN backbone for MRI feature extraction"""
    def __init__(self, in_channel=1, feat_dim=1024, expansion=4, 
                 type_name='conv3x3x3', norm_type='Instance'):
        super(NetWork, self).__init__()
        
        self.conv = nn.Sequential()
        
        self.conv.add_module('conv0_s1', nn.Conv3d(in_channel, 4*expansion, kernel_size=1))
        if norm_type == 'Instance':
            self.conv.add_module('lrn0_s1', nn.InstanceNorm3d(4*expansion))
        else:
            self.conv.add_module('lrn0_s1', nn.BatchNorm3d(4*expansion))
        self.conv.add_module('relu0_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool0_s1', nn.MaxPool3d(kernel_size=3, stride=2))
        
        self.conv.add_module('conv1_s1', nn.Conv3d(4*expansion, 32*expansion, 
                                                   kernel_size=3, padding=0, dilation=2))
        if norm_type == 'Instance':
            self.conv.add_module('lrn1_s1', nn.InstanceNorm3d(32*expansion))
        else:
            self.conv.add_module('lrn1_s1', nn.BatchNorm3d(32*expansion))
        self.conv.add_module('relu1_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool1_s1', nn.MaxPool3d(kernel_size=3, stride=2))
        
        self.conv.add_module('conv2_s1', nn.Conv3d(32*expansion, 64*expansion, 
                                                   kernel_size=5, padding=2, dilation=2))
        if norm_type == 'Instance':
            self.conv.add_module('lrn2_s1', nn.InstanceNorm3d(64*expansion))
        else:
            self.conv.add_module('lrn2_s1', nn.BatchNorm3d(64*expansion))
        self.conv.add_module('relu2_s1', nn.ReLU(inplace=True))
        
        self.conv.add_module('conv3_s1', nn.Conv3d(64*expansion, 64*expansion, 
                                                   kernel_size=3, padding=1, dilation=2))
        if norm_type == 'Instance':
            self.conv.add_module('lrn3_s1', nn.InstanceNorm3d(64*expansion))
        else:
            self.conv.add_module('lrn3_s1', nn.BatchNorm3d(64*expansion))
        self.conv.add_module('relu3_s1', nn.ReLU(inplace=True))
        self.conv.add_module('pool3_s1', nn.MaxPool3d(kernel_size=5, stride=2))
        
        self.fc6 = nn.Sequential(
            nn.Linear(64*expansion*6*6*6, feat_dim)
        )
        
        self.age_encoder = AgeEncoding(512, 0.1, feat_dim)
    
    def forward(self, x, age_id):
        z = self.conv(x)
        z = self.fc6(z.view(x.shape[0], -1))
        if age_id is not None:
            z = self.age_encoder(z, age_id)
        return z


class Flatten(nn.Module):
    def __init__(self):
        super(Flatten, self).__init__()
    
    def forward(self, feat):
        return feat.view(feat.size(0), -1)


class LinearClassifierAlexNet(nn.Module):
    """2-layer classifier for AD/MCI/CN classification"""
    def __init__(self, in_dim, n_hid=200, n_label=3):
        super(LinearClassifierAlexNet, self).__init__()
        
        self.classifier = nn.Sequential(
            Flatten(),
            nn.Linear(in_dim, n_hid),
            nn.Linear(n_hid, n_label)
        )
        self.initilize()
    
    def initilize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                m.weight.data.normal_(0, 0.01)
                m.bias.data.fill_(0.0)
    
    def forward(self, x):
        return self.classifier(x)


class alex_net_complete(nn.Module):
    """Complete model: backbone + classifier"""
    def __init__(self, image_embedding_model, classifier=None):
        super(alex_net_complete, self).__init__()
        self.image_embedding_model = image_embedding_model
        self.classifier = classifier
    
    def forward(self, input_image_variable, age_id=None):
        image_embedding = self.image_embedding_model(input_image_variable, age_id)
        if self.classifier is None:
            logit_res = image_embedding
        else:
            logit_res = self.classifier(image_embedding)
        return logit_res


def build_model(config):
    """Build model from config"""
    in_channel = config['model']['input_channel']
    feat_dim = config['model']['feature_dim']
    n_label = config['model']['n_label']
    n_hid_main = config['model']['nhid']
    expansion = config['model']['expansion']
    norm_type = config['model']['norm_type']
    
    image_embeding_model = NetWork(
        in_channel=in_channel, 
        feat_dim=feat_dim, 
        expansion=expansion,
        norm_type=norm_type
    )
    
    classifier = LinearClassifierAlexNet(in_dim=feat_dim, n_hid=n_hid_main, n_label=n_label)
    
    main_model = alex_net_complete(image_embeding_model, classifier)
    
    if config['training_parameters']['pretrain'] is not None:
        best_model_dir = './saved_model/'
        pretrained_dict = torch.load(
            best_model_dir + config['training_parameters']['pretrain'] + '_model_low_loss.pth.tar',
            map_location='cpu'
        )['state_dict']
        model_dict = main_model.state_dict()
        pretrained_dict = {k[6:]: v for k, v in pretrained_dict.items() if (k[6:] in model_dict.keys())}
        model_dict.update(pretrained_dict)
        main_model.load_state_dict(model_dict)
        print("Pretrained weights loaded!")
    
    return main_model

print("✓ Model architecture loaded")

✓ Model architecture loaded


## 7. CSV 기반 Dataset Loader (Your Data)

실제 데이터 구조에 맞게 수정된 Dataset 클래스입니다.

In [ ]:
class ADNI_CSV_Dataset(data.Dataset):
    """ADNI Dataset from CSV with pre-matched file paths"""
    
    def __init__(self, csv_data, dir_to_scans, mode='Train', n_label=2, class_filter=None):
        """
        Args:
            csv_data: pandas DataFrame with 'matched_filepath' column
            dir_to_scans: Not used (kept for compatibility)
            mode: 'Train', 'Val', 'Test'
            n_label: 2 for binary classification
            class_filter: List of classes to include (e.g., ['MCI', 'AD', 'Dementia'])
        """
        # Create dynamic label mapping based on class_filter
        if class_filter is None:
            class_filter = ['CN', 'MCI', 'AD', 'Dementia']
        
        LABEL_MAPPING = {}
        label_idx = 0
        for cls in ['CN', 'MCI', 'AD', 'Dementia']:
            if cls in class_filter:
                # Map Dementia to same label as AD if both are in filter
                if cls == 'Dementia' and 'AD' in class_filter:
                    LABEL_MAPPING[cls] = LABEL_MAPPING['AD']
                else:
                    LABEL_MAPPING[cls] = label_idx
                    label_idx += 1
        
        self.LABEL_MAPPING = LABEL_MAPPING
        self.mode = mode
        self.class_filter = class_filter
        self.age_range = list(np.arange(0.0, 120.0, 0.5))
        
        valid_data = []
        for idx, row in csv_data.iterrows():
            dx = row.get('DX', row.get('Group-RS:InitialDX', None))
            if pd.notna(dx) and dx in LABEL_MAPPING and pd.notna(row.get('matched_filepath')):
                valid_data.append(row)
        
        self.data = pd.DataFrame(valid_data)
        print(f"[{mode}] Loaded {len(self.data)} samples")
        
        if len(self.data) > 0:
            dx_col = 'DX' if 'DX' in self.data.columns else 'Group-RS:InitialDX'
            for label_name in class_filter:
                if label_name == "Dementia":
                    continue  # Skip Dementia as it's grouped with AD
                if label_name == "AD":
                    count = ((self.data[dx_col] == "AD") | (self.data[dx_col] == "Dementia")).sum()
                else:
                    count = (self.data[dx_col] == label_name).sum()
                print(f"  - {label_name}: {count} samples")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        try:
            row = self.data.iloc[idx]
            
            file_path = row['matched_filepath']
            
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"File not found: {file_path}")
            
            dx = row.get('DX', row.get('Group-RS:InitialDX'))
            label = self.LABEL_MAPPING[dx]
            
            mmse = row.get('MMSE', 0)
            if pd.isna(mmse):
                mmse = 0
            
            age = row.get('Age', 70)
            if pd.isna(age):
                age = 70
            
            age_rounded = round(age * 2) / 2
            age_rounded = max(0, min(119.5, age_rounded))
            age_idx = self.age_range.index(age_rounded)
            
            subject_id = row.get('Subject', 'unknown')
            idx_out = hash(subject_id) % 10000
            
            image = nib.load(file_path).get_fdata().squeeze()
            image[np.isnan(image)] = 0.0
            
            if image.max() > image.min():
                image = (image - image.min()) / (image.max() - image.min() + 1e-6)
            
            if self.mode == 'Train':
                sigma = np.random.uniform(0.0, 1.0, 1)[0]
                image = scipy.ndimage.gaussian_filter(image, sigma, truncate=8)
            
            image = np.expand_dims(image, axis=0)
            
            if self.mode == 'Train':
                image = self.randomCrop(image, 96, 96, 96)
            else:
                image = self.centerCrop(image, 96, 96, 96)
            
            return image.astype(np.float32), label, idx_out, mmse, 0, age_idx
        
        except Exception as e:
            print(f"Failed to load #{idx}: {e}")
            print(traceback.format_exc())
            dummy_image = np.zeros((1, 96, 96, 96), dtype=np.float32)
            return dummy_image, -100, 0, 0, 0, 0
    
    def centerCrop(self, img, length, width, height):
        """Center crop to target size"""
        if img.shape[1] < length or img.shape[2] < width or img.shape[3] < height:
            pad_x = max(0, length - img.shape[1])
            pad_y = max(0, width - img.shape[2])
            pad_z = max(0, height - img.shape[3])
            img = np.pad(img, ((0,0), (pad_x//2, pad_x-pad_x//2), 
                              (pad_y//2, pad_y-pad_y//2), (pad_z//2, pad_z-pad_z//2)), 
                        mode='constant')
        
        x = img.shape[1] // 2 - length // 2
        y = img.shape[2] // 2 - width // 2
        z = img.shape[3] // 2 - height // 2
        img = img[:, x:x+length, y:y+width, z:z+height]
        return img
    
    def randomCrop(self, img, length, width, height):
        """Random crop to target size"""
        if img.shape[1] < length or img.shape[2] < width or img.shape[3] < height:
            pad_x = max(0, length - img.shape[1])
            pad_y = max(0, width - img.shape[2])
            pad_z = max(0, height - img.shape[3])
            img = np.pad(img, ((0,0), (pad_x//2, pad_x-pad_x//2), 
                              (pad_y//2, pad_y-pad_y//2), (pad_z//2, pad_z-pad_z//2)), 
                        mode='constant')
        
        x = random.randint(0, max(0, img.shape[1] - length))
        y = random.randint(0, max(0, img.shape[2] - width))
        z = random.randint(0, max(0, img.shape[3] - height))
        img = img[:, x:x+length, y:y+width, z:z+height]
        return img

print("✓ CSV-based Dataset loader loaded (with class filtering)")

## 8. 데이터 로드 및 Train/Val/Test Split

In [ ]:
import re
import glob

def load_and_split_data(csv_file, base_path, class_filter, train_split=0.7, val_split=0.15, test_split=0.15, random_state=42):
    """
    CSV file and match with actual MRI files using S#####_I###### pattern
    Filter by specified classes for binary classification
    """
    print(f"\nLoading CSV file: {csv_file}")
    df = pd.read_csv(csv_file, low_memory=False)
    print(f"Total rows in CSV: {len(df)}")
    
    dx_col = 'DX' if 'DX' in df.columns else 'Group-RS:InitialDX'
    print(f"Using diagnosis column: {dx_col}")
    
    # Filter by specified classes only
    valid_dx = df[df[dx_col].isin(class_filter)].copy()
    print(f"Rows with diagnosis in {class_filter}: {len(valid_dx)}")
    
    print("\nDiagnosis distribution:")
    print(valid_dx[dx_col].value_counts())
    
    print("\n" + "="*60)
    print("Searching for MRI files in all ADNI_Download folders...")
    print("="*60)
    
    folders = [
        'ADNI_Download',
        'ADNI_Download 2',
        'ADNI_Download 3',
        'ADNI_Download 4',
        'ADNI_Download 5',
        'ADNI_Download 6',
        'ADNI_Download 7'
    ]
    
    all_files = []
    for folder in folders:
        folder_path = os.path.join(base_path, folder)
        if os.path.exists(folder_path):
            files = glob.glob(os.path.join(folder_path, '*.nii.gz'))
            all_files.extend(files)
            print(f"  {folder}: {len(files)} files")
    
    print(f"\nTotal MRI files found: {len(all_files)}")
    
    file_info = []
    for filepath in all_files:
        filename = os.path.basename(filepath)
        pattern = r'(S\d+)_(I\d+)'
        match = re.search(pattern, filename)
        
        if match:
            s_id = match.group(1)
            i_id = match.group(2)
            si_pattern = f"{s_id}_{i_id}"
        else:
            s_id = None
            i_id = None
            si_pattern = None
        
        file_info.append({
            'filepath': filepath,
            'filename': filename,
            's_id': s_id,
            'i_id': i_id,
            'si_pattern': si_pattern
        })
    
    file_df = pd.DataFrame(file_info)
    print(f"Parsed {len(file_df)} files with S#####_I###### pattern")
    
    valid_dx['si_pattern'] = None
    valid_dx['matched_filepath'] = None
    
    for idx, row in valid_dx.iterrows():
        new_path = row['New_Path']
        pattern = r'(S\d+)_(I\d+)'
        match = re.search(pattern, new_path)
        
        if match:
            s_id = match.group(1)
            i_id = match.group(2)
            si_pattern = f"{s_id}_{i_id}"
            valid_dx.at[idx, 'si_pattern'] = si_pattern
    
    print(f"\nExtracted S#####_I###### patterns from {valid_dx['si_pattern'].notna().sum()} rows")
    
    matched_count = 0
    for idx, row in valid_dx.iterrows():
        si_pattern = row['si_pattern']
        
        if pd.notna(si_pattern):
            match = file_df[file_df['si_pattern'] == si_pattern]
            
            if len(match) > 0:
                valid_dx.at[idx, 'matched_filepath'] = match.iloc[0]['filepath']
                matched_count += 1
    
    print(f"Matched {matched_count} out of {len(valid_dx)} sessions")
    
    matched_only = valid_dx[valid_dx['matched_filepath'].notna()].copy()
    print(f"\nFiltered to matched sessions only: {len(matched_only)}")
    print("Matched diagnosis distribution:")
    print(matched_only[dx_col].value_counts())
    
    train_data, temp_data = train_test_split(
        matched_only, 
        train_size=train_split, 
        stratify=matched_only[dx_col],
        random_state=random_state
    )
    
    val_ratio = val_split / (val_split + test_split)
    val_data, test_data = train_test_split(
        temp_data,
        train_size=val_ratio,
        stratify=temp_data[dx_col],
        random_state=random_state
    )
    
    print(f"\n✅ Data split complete:")
    print(f"  - Train: {len(train_data)} samples ({len(train_data)/len(matched_only)*100:.1f}%)")
    print(f"  - Val: {len(val_data)} samples ({len(val_data)/len(matched_only)*100:.1f}%)")
    print(f"  - Test: {len(test_data)} samples ({len(test_data)/len(matched_only)*100:.1f}%)")
    
    return train_data, val_data, test_data

print("✓ Data loading function loaded (with class filtering)")

## 9. Training Functions

In [87]:
def train(cfg, train_loader, main_model, scheduler, criterion, main_optimizer, epoch):
    """Train for one epoch"""
    batch_time = AverageMeter()
    data_time = AverageMeter()
    main_losses = AverageMeter()
    
    # Switch to train mode
    main_model.train()
    end = time.time()
    
    logit_all = []
    target_all = []
    
    for i, (input, target, index, mmse, segment, age) in enumerate(train_loader):
        # Skip invalid samples
        if target[0] == -100:
            continue
        
        # Measure data loading time
        data_time.update(time.time() - end)
        
        # Move to device
        input = input.to(device)
        target = target.to(device)
        index = index.to(device)
        age = age.to(device) if cfg['training_parameters']['use_age'] else None
        
        # Forward pass
        main_loss, logit = main_model([input, age], target)
        main_loss = main_loss.mean()
        
        # Store predictions
        logit_all.append(logit.data.cpu())
        target_all.append(target.data.cpu())
        acc, _ = accuracy(logit.data.cpu(), target.data.cpu())
        
        # Backward pass
        main_optimizer.zero_grad()
        main_loss.backward()
        clip_gradients(main_model, i, cfg['training_parameters']['max_grad_l2_norm'])
        main_optimizer.step()
        
        # Record loss
        main_losses.update(main_loss.cpu().item(), input.size(0))
        
        # Measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()
        
        # Print progress
        if i % cfg['training_parameters']['print_freq'] == 0:
            print(f'Epoch: [{epoch}][{i}/{len(train_loader)}]\t'
                  f'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  f'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  f'Loss {main_losses.val:.4f} ({main_losses.avg:.4f})\t'
                  f'Accuracy {acc[0].item():.3f}')
    
    # Calculate balanced accuracy
    if len(logit_all) > 0:
        logit_all = torch.cat(logit_all).numpy()
        target_all = torch.cat(target_all).numpy()
        acc_all = balanced_accuracy_score(target_all, np.argmax(logit_all, 1))
    else:
        acc_all = 0.0
    
    return main_losses.avg, acc_all * 100


def validate(cfg, val_loader, main_model, criterion, epoch):
    """Validate the model"""
    batch_time = AverageMeter()
    data_time = AverageMeter()
    main_losses = AverageMeter()
    
    end = time.time()
    correct_all = 0.0
    
    # Switch to eval mode
    main_model.eval()
    
    confusion_mat = torch.zeros(cfg['model']['n_label'], cfg['model']['n_label'])
    logit_all = []
    target_all = []
    
    with torch.no_grad():
        for i, (input, target, patient_idx, mmse, segment, age) in enumerate(val_loader):
            # Skip invalid samples
            if target[0] == -100:
                continue
            
            # Measure data loading time
            data_time.update(time.time() - end)
            
            # Move to device
            input = input.to(device)
            target = target.to(device)
            age = age.to(device) if cfg['training_parameters']['use_age'] else None
            
            # Forward pass
            main_loss, logit = main_model([input, age], target)
            main_loss = main_loss.mean()
            
            # Store predictions
            logit_all.append(torch.tensor(logit.data.cpu()))
            target_all.append(torch.tensor(target.data.cpu()))
            
            # Calculate accuracy
            acc, _ = accuracy(logit.data.cpu(), target.data.cpu())
            _, preds = torch.max(logit.cpu(), 1)
            
            # Update confusion matrix
            for t, p in zip(target.cpu().view(-1), preds.view(-1)):
                confusion_mat[t.long(), p.long()] += 1
            
            acc, correct = accuracy(logit.cpu(), target.cpu())
            correct_all += correct[0].item()
            
            # Record loss
            main_losses.update(main_loss.cpu().item(), input.size(0))
            
            # Measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()
            
            # Print progress
            if i % cfg['training_parameters']['print_freq'] == 0:
                print(f'Validation [{i}/{len(val_loader)}]\t'
                      f'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                      f'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                      f'Loss {main_losses.val:.4f} ({main_losses.avg:.4f})\t'
                      f'Accuracy {acc[0].item():.3f}')
    
    # Calculate metrics
    if len(logit_all) > 0:
        logit_all = torch.cat(logit_all).numpy()
        target_all = torch.cat(target_all).numpy()
        acc_all = balanced_accuracy_score(target_all, np.argmax(logit_all, 1))
        plotting_fpr, plotting_tpr, roc_auc = get_auc_data(logit_all, target_all, cfg['model']['n_label'])
    else:
        acc_all = 0.0
        plotting_fpr, plotting_tpr, roc_auc = [], [], {}
    
    return main_losses.avg, acc_all * 100, confusion_mat, [plotting_fpr, plotting_tpr, roc_auc]


def save_checkpoint(state, is_best, lowest_loss, is_best_micro_auc, is_best_macro_auc, 
                    is_best_auc, filename='checkpoint.pth.tar'):
    """Save model checkpoint"""
    saving_dir = Path(filename).parent
    if not os.path.exists(saving_dir):
        os.makedirs(saving_dir)
    
    torch.save(state, filename)
    
    if is_best:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best.pth.tar')
    if lowest_loss:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_low_loss.pth.tar')
    if is_best_micro_auc:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best_micro.pth.tar')
    if is_best_macro_auc:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best_macro.pth.tar')
    if is_best_auc:
        shutil.copyfile(filename, filename.replace('.pth.tar', '') + '_model_best_auc.pth.tar')

print("✓ Training functions loaded")

✓ Training functions loaded


## 10. Main Training Loop

In [ ]:
def main(cfg):
    """Main training function"""
    best_prec1 = 0
    best_loss = 1000
    best_micro_auc = 0
    best_macro_auc = 0
    
    seed = 168
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.device_count() > 0:
        torch.cuda.manual_seed(seed)
    
    print("\n" + "="*60)
    print("Building model...")
    print("="*60)
    
    main_model = build_model(cfg)
    main_model = main_model.to(device)
    print(f"Model built successfully! Parameters: {sum(p.numel() for p in main_model.parameters())/1e6:.2f}M")
    
    criterion = get_loss_criterion(cfg, type='CrossEntropyLoss').to(device)
    
    model = DataParallel_withLoss(main_model, criterion)
    
    if hasattr(model, 'module'):
        print('Has module!')
        model = model.module
    
    params = [{'params': filter(lambda p: p.requires_grad, model.parameters())}]
    main_optim = getattr(optim, cfg['optimizer']['method'])(params, **cfg['optimizer']['par'])
    
    scheduler = MultiStepLR(main_optim, milestones=[20, 50], gamma=0.1)
    
    print("\n" + "="*60)
    print("Loading data...")
    print("="*60)
    
    train_data, val_data, test_data = load_and_split_data(
        cfg['data']['csv_file'],
        cfg['data']['base_path'],
        class_filter=cfg['data']['classes'],
        train_split=cfg['data']['train_split'],
        val_split=cfg['data']['val_split'],
        test_split=cfg['data']['test_split']
    )
    
    train_dataset = ADNI_CSV_Dataset(
        train_data, None, mode='Train', n_label=cfg['model']['n_label'], 
        class_filter=cfg['data']['classes']
    )
    val_dataset = ADNI_CSV_Dataset(
        val_data, None, mode='Val', n_label=cfg['model']['n_label'],
        class_filter=cfg['data']['classes']
    )
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=cfg['data']['batch_size'],
        shuffle=True, num_workers=cfg['data']['workers'],
        pin_memory=True, drop_last=True
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=cfg['data']['val_batch_size'],
        shuffle=False, num_workers=cfg['data']['workers'],
        pin_memory=True
    )
    
    print(f'Training batches: {len(train_loader)}')
    print(f'Validation batches: {len(val_loader)}')
    
    print("\n" + "="*60)
    print("Starting training...")
    print("="*60 + "\n")
    
    for epoch in range(cfg['training_parameters']['start_epoch'], cfg['training_parameters']['epochs']):
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}/{cfg['training_parameters']['epochs']}]")
        print(f"{'='*60}")
        
        train_loss, train_acc = train(cfg, train_loader, model, scheduler, criterion, main_optim, epoch)
        
        val_loss, val_acc, confusion_matrix, auc_outs = validate(cfg, val_loader, model, criterion, epoch)
        
        scheduler.step()
        
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}] Summary:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        if len(auc_outs[2]) > 0:
            micro_auc = auc_outs[2].get(len(auc_outs[2])-2, 0)
            macro_auc = auc_outs[2].get(len(auc_outs[2])-1, 0)
            print(f"  Micro AUC: {micro_auc:.4f}")
            print(f"  Macro AUC: {macro_auc:.4f}")
        print(f"{'='*60}\n")
        
        is_best = (val_acc > best_prec1)
        lowest_loss = (val_loss < best_loss)
        
        if len(auc_outs[2]) > 0:
            micro_auc = auc_outs[2].get(len(auc_outs[2])-2, 0)
            macro_auc = auc_outs[2].get(len(auc_outs[2])-1, 0)
            is_best_micro_auc = (micro_auc >= best_micro_auc)
            is_best_macro_auc = (macro_auc > best_macro_auc)
            is_best_auc = (micro_auc > 0.8) & (macro_auc > 0.8)
        else:
            is_best_micro_auc = False
            is_best_macro_auc = False
            is_best_auc = False
            micro_auc = 0
            macro_auc = 0
        
        best_prec1 = max(val_acc, best_prec1)
        best_loss = min(val_loss, best_loss)
        best_micro_auc = max(micro_auc, best_micro_auc)
        best_macro_auc = max(macro_auc, best_macro_auc)
        
        save_checkpoint(
            {
                'epoch': epoch + 1,
                'state_dict': model.state_dict(),
                'best_prec1': best_prec1,
                'best_loss': best_loss,
                'optimizer': main_optim.state_dict(),
            },
            is_best, lowest_loss, is_best_micro_auc, is_best_macro_auc, is_best_auc,
            filename=cfg['file_name'] + '.pth.tar'
        )
        
        if is_best:
            print(f"New best accuracy: {val_acc:.2f}%")
        if is_best_micro_auc:
            print(f"New best micro AUC: {micro_auc:.4f}")
        if is_best_macro_auc:
            print(f"New best macro AUC: {macro_auc:.4f}")
    
    print("\n" + "="*60)
    print("Training completed!")
    print(f"Best Accuracy: {best_prec1:.2f}%")
    print(f"Best Micro AUC: {best_micro_auc:.4f}")
    print(f"Best Macro AUC: {best_macro_auc:.4f}")
    print("="*60)

print("✓ Main training loop loaded (for binary classification)")

## 11. 데이터 확인 (실행 전 체크)

In [89]:
import glob

print("Checking data paths...\n")

csv_file = cfg['data']['csv_file']
if os.path.exists(csv_file):
    print(f"CSV file exists: {csv_file}")
    df = pd.read_csv(csv_file, low_memory=False)
    print(f"  Total rows: {len(df)}")
    
    dx_col = 'DX' if 'DX' in df.columns else 'Group-RS:InitialDX'
    if dx_col in df.columns:
        print(f"  Diagnosis distribution:")
        print(df[dx_col].value_counts())
else:
    print(f"CSV file NOT found: {csv_file}")

base_path = cfg['data']['base_path']
print(f"\nBase path: {base_path}")

folders = [
    'ADNI_Download',
    'ADNI_Download 2',
    'ADNI_Download 3',
    'ADNI_Download 4',
    'ADNI_Download 5',
    'ADNI_Download 6',
    'ADNI_Download 7'
]

total_files = 0
for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        nii_files = glob.glob(os.path.join(folder_path, '*.nii.gz'))
        total_files += len(nii_files)
        print(f"  {folder}: {len(nii_files)} files")

print(f"\nTotal MRI files across all folders: {total_files}")

print("\n" + "="*60)
print("Ready to train!")
print("="*60)

Checking data paths...

CSV file exists: /Users/othree/Cognitive Reserve Modeling/Data/ADNI_master_merged_12-17-2025.csv
  Total rows: 4508
  Diagnosis distribution:
DX
CN          1737
MCI         1476
Dementia    1295
Name: count, dtype: int64

Base path: /Users/othree/Cognitive Reserve Modeling/Data
  ADNI_Download: 1941 files

Total MRI files across all folders: 1941

Ready to train!


## 12. 🚀 학습 시작!

아래 셀을 실행하면 학습이 시작됩니다.

In [90]:
# 학습 시작!
if __name__ == '__main__':
    main(cfg)


Building model...
Model built successfully! Parameters: 138.25M
Has module!

Loading data...

Loading CSV file: /Users/othree/Cognitive Reserve Modeling/Data/ADNI_master_merged_12-17-2025.csv
Total rows in CSV: 4508
Using diagnosis column: DX
Rows with valid diagnosis (CN/MCI/AD/Dementia): 4508

Diagnosis distribution:
DX
CN          1737
MCI         1476
Dementia    1295
Name: count, dtype: int64

Searching for MRI files in all ADNI_Download folders...
  ADNI_Download: 1941 files

Total MRI files found: 1941
Parsed 1941 files with S#####_I###### pattern

Extracted S#####_I###### patterns from 4487 rows
Matched 1572 out of 4508 sessions

Filtered to matched sessions only: 1572
Matched diagnosis distribution:
DX
MCI         588
CN          509
Dementia    475
Name: count, dtype: int64

✅ Data split complete:
  - Train: 1100 samples (70.0%)
  - Val: 236 samples (15.0%)
  - Test: 236 samples (15.0%)
[Train] Loaded 1100 samples
  - CN: 356 samples
  - MCI: 412 samples
  - AD: 332 samples


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch: [0][0/275]	Time 12.250 (12.250)	Data 0.764 (0.764)	Loss 1.1658 (1.1658)	Accuracy 0.000


KeyboardInterrupt: 

## 13. 학습된 모델 로드 및 평가

In [ ]:
# 학습된 모델 로드
def load_trained_model(checkpoint_path, cfg):
    """Load trained model from checkpoint"""
    model = build_model(cfg)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Remove 'module.' prefix if exists
    state_dict = checkpoint['state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    
    model.load_state_dict(new_state_dict)
    model = model.to(device)
    model.eval()
    
    print(f"Model loaded from {checkpoint_path}")
    print(f"Epoch: {checkpoint['epoch']}")
    print(f"Best accuracy: {checkpoint['best_prec1']:.2f}%")
    
    return model

# 예시: 최고 정확도 모델 로드
# model = load_trained_model('./saved_model/adni_training_model_best.pth.tar', cfg)

## 14. 하이퍼파라미터 조정 가이드

### 주요 하이퍼파라미터:

1. **Expansion (모델 너비)**:
   ```python
   cfg['model']['expansion'] = 8  # 기본값 (논문 최적값)
   # 작은 데이터셋: 4
   # 큰 데이터셋: 16
   ```

2. **Learning Rate**:
   ```python
   cfg['optimizer']['par']['lr'] = 0.01  # 기본값
   # 불안정한 학습: 0.001
   ```
   
3. **Batch Size**:
   ```python
   cfg['data']['batch_size'] = 4  # GPU 메모리에 따라 조정
   ```

4. **Epochs**:
   ```python
   cfg['training_parameters']['epochs'] = 100  # 기본값
   # 빠른 실험: 20-50
   # 완전 수렴: 200
   ```

5. **Age Encoding**:
   ```python
   cfg['training_parameters']['use_age'] = True  # 나이 정보 사용
   # 약 2% 성능 향상 기대
   ```

### 메모리 부족 시:
```python
cfg['data']['batch_size'] = 2
cfg['data']['val_batch_size'] = 1
cfg['model']['expansion'] = 4
cfg['data']['workers'] = 0
```

## 15. 예상 결과 (논문 기준)

### ADNI Test Set:
- **Accuracy**: ~68%
- **Balanced Accuracy**: ~70%
- **Micro AUC**: ~82%
- **Macro AUC**: ~80%

### 클래스별 AUC:
- **CN (정상)**: ~88%
- **MCI (경도인지장애)**: ~63%
- **AD (알츠하이머)**: ~89%

### 학습 시간 예상:
- MacBook (Apple Silicon): ~8-12시간 (100 epochs)
- Colab GPU: ~6-8시간 (100 epochs)

---

**데이터 경로가 올바르게 설정되어 있으니 바로 실행 가능합니다!** ✅